# Fraud Detection — PaySim Synthetic Financial Dataset

This project builds and compares fraud-detection models on the [PaySim synthetic mobile money dataset](https://www.kaggle.com/datasets/ealaxi/paysim1) (~6.3M transactions, ~0.13% fraud). It covers data cleaning, feature engineering, handling extreme class imbalance, and comparing three model types — including a real investigation into a data-leakage red flag along the way.


**Tools:** `pandas`, `scikit-learn`, `xgboost` (GPU-accelerated via CUDA)


## Part 1: Setup & Load Data

Load the raw dataset and get a first look at its size and structure.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve

In [26]:
df = pd.read_csv("PS_20174392719_1491204439457_log.csv")
df.info()
df.head()
df['isFraud'].value_counts()

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\mohma\AppData\Local\Temp\ipykernel_29932\2944446435.py:1: SyntaxWarning: invalid escape sequence '\C'
  df = pd.read_csv("C:\CURSOR PROGRAM FILES\VCODES\TRAIL STUFF\FraudProj\PS_20174392719_1491204439457_log.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


isFraud
0    6354407
1       8213
Name: count, dtype: int64

**Finding:** 6,362,620 transactions across 11 columns, no missing values apparent. Only **8,213 (0.13%)** are fraudulent — an extreme class imbalance that shapes every decision in this project. A model that always predicts "not fraud" would already be 99.87% accurate while catching zero fraud — which is why accuracy alone is the wrong metric here.

## Part 2: Data Quality Checks

Confirm there are no missing values, inspect the transaction types present, and check for duplicate rows.

In [27]:
df.isnull().sum(),  

(step              0
 type              0
 amount            0
 nameOrig          0
 oldbalanceOrg     0
 newbalanceOrig    0
 nameDest          0
 oldbalanceDest    0
 newbalanceDest    0
 isFraud           0
 isFlaggedFraud    0
 dtype: int64,)

In [28]:
df['type'].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [29]:
df.duplicated().sum()

KeyboardInterrupt: 

**Finding:** No nulls, no duplicates, 5 transaction types (CASH_OUT, PAYMENT, CASH_IN, TRANSFER, DEBIT) with reasonable amounts — the data is clean going in.

## Part 3: Where Does Fraud Actually Occur?

Check whether fraud is spread across all transaction types, or concentrated in specific ones.

In [ ]:
df.groupby('type')['isFraud'].sum()

type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64

**Finding:** Fraud occurs **only** in `CASH_OUT` (4,116 cases) and `TRANSFER` (4,097 cases) — zero fraud exists in CASH_IN, DEBIT, or PAYMENT, despite those making up the majority of transactions. This is a structural property of the dataset, not a coincidence — so the analysis is narrowed to these two transaction types, cutting out noise without losing a single fraud case.

## Part 4: Filter to Fraud-Relevant Transaction Types

In [ ]:
df_filtered = df[df['type'].isin(['CASH_OUT', 'TRANSFER'])].copy()
df_filtered.shape

(2770409, 11)

**Result:** 2,770,409 rows retained (all 8,213 fraud cases preserved), down from 6.36M — a much more focused dataset to work with.

## Part 5: Feature Engineering — Balance Discrepancies

For a legitimate transaction, the sender's balance should drop by roughly the transaction amount, and the receiver's balance should rise by roughly the same amount. Two features are engineered to capture how far actual balances deviate from this expectation.

In [ ]:
df_filtered['balance_diff_orig'] = df_filtered['oldbalanceOrg'] - df_filtered['amount'] - df_filtered['newbalanceOrig']

In [ ]:
df_filtered.groupby('isFraud')['balance_diff_orig'].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,2762196.0,-286803.509954,876375.192156,-92445516.64,-280466.255,-144200.825,-52613.43,1.000000e-02
1,8213.0,-10692.325265,265146.131130,-10000000.00,0.000,0.000,0.00,3.725290e-09


**Finding:** Counterintuitively, legitimate transactions show large, messy balance discrepancies (median ≈ -144,200), while fraudulent transactions are almost perfectly consistent (median = 0.000). This turned out to be a strong — arguably *too* strong — predictive signal, investigated further in Part 8.

In [ ]:
df_filtered['balance_diff_dest'] = df_filtered['oldbalanceDest'] + df_filtered['amount'] - df_filtered['newbalanceDest']

In [ ]:
df_filtered.groupby('isFraud')['balance_diff_dest'].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,2762196.0,-30910.325352,5.840943e+05,-75885725.63,0.0,0.00,0.00,9977761.06
1,8213.0,732509.301069,1.867748e+06,-8875516.29,0.0,2231.46,442722.01,10000000.00


**Finding:** A similar pattern on the destination side — legitimate transactions typically show zero discrepancy, while fraud cases often show a positive gap (median ≈ 2,231), meaning the amount that left the sender didn't fully show up in the receiver's new balance. Together, both features paint a picture of fraud as *"money leaves cleanly, but doesn't arrive properly"* in this dataset.

## Part 6: Encode & Prepare Features for Modeling

Encode the categorical `type` column and drop columns that shouldn't be fed to the model: account ID strings (`nameOrig`, `nameDest`) carry no learnable pattern, and `isFlaggedFraud` is another system's rule-based output, not a legitimate model input.

In [ ]:
df_filtered['type_encoded'] = df_filtered['type'].apply(lambda x: 1 if x == 'TRANSFER' else 0)

In [ ]:
df_model = df_filtered.drop(columns=['nameOrig', 'nameDest', 'type', 'isFlaggedFraud'])
df_model.columns

Index(['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
       'newbalanceDest', 'isFraud', 'balance_diff_orig', 'balance_diff_dest',
       'type_encoded'],
      dtype='object')

## Part 7: Train/Test Split

With fraud at only ~0.3% of the data, the split is stratified to preserve the same fraud ratio in both the training and test sets — otherwise the test set could end up with too few fraud cases to evaluate reliably.

In [ ]:
X = df_model.drop(columns=['isFraud'])
y = df_model['isFraud']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

isFraud
0    0.997035
1    0.002965
Name: proportion, dtype: float64

## Part 8: Baseline Model — Logistic Regression

`class_weight='balanced'` is used to force the model to pay attention to the rare fraud class, rather than defaulting to always predicting "not fraud."

In [ ]:
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000)
log_reg.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [ ]:
y_pred = log_reg.predict(X_test)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, log_reg.predict_proba(X_test)[:, 1]))

              precision    recall  f1-score   support

           0       1.00      0.89      0.94    552439
           1       0.02      0.95      0.05      1643

    accuracy                           0.89    554082
   macro avg       0.51      0.92      0.49    554082
weighted avg       1.00      0.89      0.94    554082

ROC-AUC: 0.973610484257705


In [ ]:
print(confusion_matrix(y_test, y_pred))

[[489711  62728]
 [    81   1562]]


**Finding:** Recall on fraud is strong (95% — only 81 of 1,643 fraud cases missed), but precision is extremely low (2%) — 62,728 legitimate transactions were wrongly flagged as fraud. ROC-AUC of 0.974 shows the model *can* rank transactions well, but the default probability threshold combined with aggressive class weighting produces far too many false alarms for practical use.

## Part 9: Random Forest

A more flexible model that can capture non-linear patterns, tried with the same class balancing approach.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [ ]:
y_pred_rf = rf.predict(X_test)

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))
print(confusion_matrix(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552439
           1       1.00      1.00      1.00      1643

    accuracy                           1.00    554082
   macro avg       1.00      1.00      1.00    554082
weighted avg       1.00      1.00      1.00    554082

ROC-AUC: 0.9990852461374581
[[552438      1]
 [     5   1638]]


**Result:** Near-perfect scores — 100% precision, 100% recall, only 1 false positive and 5 false negatives out of 554,082 test transactions. This is suspicious rather than celebratory: a jump this large from Logistic Regression is a classic sign of data leakage, investigated next rather than taken at face value.

## Part 10: Investigating the Suspiciously Perfect Result

Checking feature importances to see what the model is actually relying on.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False)

balance_diff_orig    0.424804
oldbalanceOrg        0.229927
newbalanceOrig       0.116698
newbalanceDest       0.070017
balance_diff_dest    0.049166
amount               0.041100
oldbalanceDest       0.031214
step                 0.026225
type_encoded         0.010849
dtype: float64

**Finding:** `balance_diff_orig` alone accounts for **42%** of the model's decision-making — more than double the next most important feature. This feature was built directly from a pattern observed in this specific simulated dataset (fraud cases cluster at a discrepancy of almost exactly 0), and Random Forest appears to have learned a near-deterministic rule off it. This is very likely an artifact of how PaySim generates its synthetic fraud rows, rather than a fraud pattern that would generalize to real-world data — so the model is retrained without it to get an honest read on performance.

## Part 11: Random Forest — Retrained Without the Leaky Feature

In [30]:
X_train_v2 = X_train.drop(columns=['balance_diff_orig'])
X_test_v2 = X_test.drop(columns=['balance_diff_orig'])

rf_v2 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_v2.fit(X_train_v2, y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [31]:
y_pred_rf_v2 = rf_v2.predict(X_test_v2)

print(classification_report(y_test, y_pred_rf_v2))
print("ROC-AUC:", roc_auc_score(y_test, rf_v2.predict_proba(X_test_v2)[:, 1]))
print(confusion_matrix(y_test, y_pred_rf_v2))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552439
           1       0.94      0.82      0.88      1643

    accuracy                           1.00    554082
   macro avg       0.97      0.91      0.94    554082
weighted avg       1.00      1.00      1.00    554082

ROC-AUC: 0.9974639342862889
[[552351     88]
 [   292   1351]]


**Result:** A much more credible outcome — 94% precision, 82% recall, 88 false positives, 292 missed fraud cases, ROC-AUC 0.997. This is the version of Random Forest that should actually be trusted, and it's a meaningful improvement in precision over Logistic Regression (94% vs. 2%) while still catching the large majority of fraud.

## Part 12: XGBoost (GPU-Accelerated)

A gradient-boosted alternative, trained on GPU (NVIDIA CUDA) using the same leak-free feature set. Class imbalance is handled via `scale_pos_weight`, XGBoost's equivalent of `class_weight='balanced'`.

In [32]:
import sys
!{sys.executable} -m pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB 330.3 kB/s eta 0:02:29
   ---------------------------------------- 0.0/48.9 MB 393.8 kB/s eta 0:02:05
   ---------------------------------------- 0.1/48.9 MB 469.7 kB/s eta 0:01:45
   ---------------------------------------- 0.1/48.9 MB 535.8 kB/s eta 0:01:32
   ---------------------------------------- 0.2/48.9 MB 892.5 kB/s eta 0:00:55
   ---------------------------------------- 0.3/48.9 MB 927.4 kB/s eta 0:00:53
   ---------------------------------------- 0.5/48.9 MB 1.6 MB/s eta 0:00:31
    --------------------------------------- 0.6/48.9 MB 1.7 MB/s eta 0:00:28
    --------------------------------------- 1.0/48.9 MB 2.5 MB/s eta 0:00:19
   - -------------------------------------- 1.5/48.9 MB 3.3 MB/s eta 0:00:15
   - -------------------------------------- 2.1/48.9 MB 4.3 MB/s eta 0:00:12
   -- ------------------------------------- 3.6/48.9 MB 6.6 MB/s eta 0:


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(scale_pos_weight)

336.34048706240486


In [34]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    device='cuda',
    tree_method='hist',
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_v2, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [35]:
y_pred_xgb = xgb_model.predict(X_test_v2)

print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, xgb_model.predict_proba(X_test_v2)[:, 1]))
print(confusion_matrix(y_test, y_pred_xgb))

c:\Users\mohma\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:569: UserWarning: [10:24:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552439
           1       0.53      0.98      0.69      1643

    accuracy                           1.00    554082
   macro avg       0.77      0.99      0.84    554082
weighted avg       1.00      1.00      1.00    554082

ROC-AUC: 0.9988520771811098
[[551020   1419]
 [    26   1617]]


**Result:** 53% precision, 98% recall — only 26 of 1,643 fraud cases missed, at the cost of 1,419 false positives. ROC-AUC of 0.999, the highest of all three models.

## Summary & Model Comparison

| Model | Precision (fraud) | Recall (fraud) | False Positives | False Negatives | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 0.02 | 0.95 | 62,728 | 81 | 0.974 |
| Random Forest (with leaky feature) | 1.00 | 1.00 | 1 | 5 | 0.999 |
| **Random Forest (honest)** | **0.94** | **0.82** | **88** | **292** | **0.997** |
| **XGBoost (honest, GPU)** | **0.53** | **0.98** | **1,419** | **26** | **0.999** |

**Key takeaways:**
- Fraud in this dataset is structurally confined to `CASH_OUT` and `TRANSFER` transactions — filtering to these two types cut the dataset size by more than half with zero loss of fraud cases.
- Two engineered balance-discrepancy features captured a genuine, if counterintuitive, fraud signal: legitimate transactions have messy balances, fraud has suspiciously clean ones.
- A near-perfect Random Forest result was investigated rather than reported at face value — `feature_importances_` revealed 42% of the model's decisions relied on a single feature that was very likely an artifact of the synthetic data generation process, not a generalizable fraud signal. Retraining without it produced a more credible, defensible result.
- **Random Forest and XGBoost represent a genuine business tradeoff**: Random Forest produces far fewer false alarms (88 vs. 1,419) but misses more fraud (292 vs. 26); XGBoost catches nearly all fraud but at a much higher false-alarm cost. Which is preferable depends on whether an organization prioritizes minimizing manual review workload or minimizing missed fraud.
